In [21]:
import numpy as np
import pandas as pd
import time

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

import joblib

### CARGAR DATASETS

In [3]:
datos = np.load("datos/gatos_perros_dataset.npz")

X_train = datos["X_train"]
y_train = datos["y_train"]

X_val = datos["X_val"]
y_val = datos["y_val"]

X_test = datos["X_test"]
y_test = datos["y_test"]


# Cargar datos de test por separado
datos_test = np.load("datos/datos-test.npz")

X_test_independiente = datos_test["X_test"]
y_test_independiente = datos_test["y_test"]


# Cargar datos de prueba individual
datos_individual = np.load("datos/datos-prueba-individual.npz")

X_individual = datos_individual["X_individual"]
y_individual = datos_individual["y_individual"]


print("Datos cargados correctamente:\n")

print(f"X_train:      {X_train.shape} | y_train:      {y_train.shape}")
print(f"X_val:        {X_val.shape} | y_val:        {y_val.shape}")
print(f"X_test:       {X_test.shape} | y_test:       {y_test.shape}")
print(f"X_test_ind.:  {X_test_independiente.shape} | y_test_ind.:  {y_test_independiente.shape}")
print(f"X_individual: {X_individual.shape} | y_individual: {y_individual.shape}")

Datos cargados correctamente:

X_train:      (13993, 784) | y_train:      (13993,)
X_val:        (2998, 784) | y_val:        (2998,)
X_test:       (2999, 784) | y_test:       (2999,)
X_test_ind.:  (2999, 784) | y_test_ind.:  (2999,)
X_individual: (10, 784) | y_individual: (10,)


### FUNCIONES AUXILIARES

In [4]:
def calcular_metricas(y_real, y_pred):

    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(
            y_real,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_real,
            y_pred,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_real,
            y_pred,
            zero_division=0
        )
    }

## CREAR CNN

In [5]:
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_val_cnn = X_val.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

In [6]:
modelo_cnn = Sequential([
    Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(28, 28, 1)
    ),

    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),

    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(128, activation="relu"),

    Dropout(0.5),

    Dense(1, activation="sigmoid")
])


modelo_cnn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


modelo_cnn.summary()

c:\Users\Jpose\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 223,873 (874.50 KB)

 Trainable params: 223,873 (874.50 KB)

 Non-trainable params: 0 (0.00 B)

### ENTRENAR CNN

In [7]:
inicio = time.perf_counter()

historial_cnn = modelo_cnn.fit(
    X_train_cnn,
    y_train,

    validation_data=(
        X_val_cnn,
        y_val
    ),

    epochs=15,
    batch_size=128,

    verbose=1
)

tiempo_entrenamiento_cnn = (time.perf_counter() - inicio)

Epoch 1/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.7788 - loss: 0.4645 - val_accuracy: 0.8399 - val_loss: 0.3634
Epoch 2/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8498 - loss: 0.3409 - val_accuracy: 0.8549 - val_loss: 0.3258
Epoch 3/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8671 - loss: 0.3089 - val_accuracy: 0.8599 - val_loss: 0.3170
Epoch 4/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8740 - loss: 0.2896 - val_accuracy: 0.8686 - val_loss: 0.3025
Epoch 5/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8848 - loss: 0.2732 - val_accuracy: 0.8672 - val_loss: 0.3031
Epoch 6/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8884 - loss: 0.2626 - val_accuracy: 0.8699 - val_loss: 0.2975
Epoch 7/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8919 - loss: 0.2531 - val_accuracy: 0.8689 - val_loss: 0.2956
Epoch 8/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8982 - loss: 0.2399 - val_accu

### PREDICCION CNN

In [8]:
inicio = time.perf_counter()

probabilidades_cnn = modelo_cnn.predict(
    X_test_cnn,
    verbose=0
)

tiempo_prediccion_cnn = (time.perf_counter() - inicio)


# Convertir probabilidades a clases
y_pred_cnn = (probabilidades_cnn.ravel() >= 0.5).astype(int)

### MÉTRICAS CNN

In [9]:
metricas_cnn = calcular_metricas(
    y_test,
    y_pred_cnn
)

matriz_cnn = confusion_matrix(
    y_test,
    y_pred_cnn
)

## CREAR MLP

In [10]:
modelo_mlp = MLPClassifier(
    hidden_layer_sizes=(8,),
    activation="logistic",
    solver="sgd",
    learning_rate_init=0.0001,
    max_iter=20,
    early_stopping=False,
    random_state=42
)

### ENTRENAR MLP

In [11]:
inicio = time.perf_counter()

modelo_mlp.fit(
    X_train,
    y_train
)

tiempo_entrenamiento_mlp = (time.perf_counter() - inicio)

c:\Users\Jpose\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


### 10. PREDICCION MLP

In [12]:
inicio = time.perf_counter()

y_pred_mlp = modelo_mlp.predict(X_test)

tiempo_prediccion_mlp = (time.perf_counter() - inicio)

### METRICAS MLP

In [13]:
metricas_mlp = calcular_metricas(
    y_test,
    y_pred_mlp
)

matriz_mlp = confusion_matrix(
    y_test,
    y_pred_mlp
)


## RESULTADOS GENERALES

In [14]:
resultados_generales_1 =  pd.DataFrame([
    {
        "modelo": "MLP",

        "accuracy": metricas_mlp["accuracy"],
        "precision": metricas_mlp["precision"],
        "recall": metricas_mlp["recall"],
        "f1_score": metricas_mlp["f1_score"],

        "tiempo_entrenamiento_s": tiempo_entrenamiento_mlp,
        "tiempo_prediccion_s": tiempo_prediccion_mlp,
        "iteraciones": modelo_mlp.n_iter_
    },

    {
        "modelo": "CNN",

        "accuracy": metricas_cnn["accuracy"],
        "precision": metricas_cnn["precision"],
        "recall": metricas_cnn["recall"],
        "f1_score": metricas_cnn["f1_score"],

        "tiempo_entrenamiento_s": tiempo_entrenamiento_cnn,
        "tiempo_prediccion_s": tiempo_prediccion_cnn,
        "iteraciones": len(historial_cnn.history["loss"])
    }
])

In [ ]:
print(resultados_generales_1.to_string(index=False))


print("\nMatriz de confusion CNN:")
print(matriz_cnn)

print("\nMatriz de confusion MLP:")
print(matriz_mlp)

modelo  accuracy  precision  recall  f1_score  tiempo_entrenamiento_s  tiempo_prediccion_s  iteraciones
   MLP  0.500167   0.500167   1.000  0.666815                1.289626             0.003364           20
   CNN  0.881627   0.855811   0.918  0.885815               28.264121             0.502617           15

Matriz de confusión CNN:
[[1267  232]
 [ 123 1377]]

Matriz de confusión MLP:
[[   0 1499]
 [   0 1500]]


## PREPARAR IMAGENES INDIVIDUALES

In [16]:
X_individual_normalizado = (X_individual.astype(np.float32) / 255.0)

X_individual_mlp = X_individual_normalizado.reshape(
    len(X_individual_normalizado),
    -1
)

X_individual_cnn = X_individual_normalizado.reshape(
    -1,
    28,
    28,
    1
)

In [17]:
resultados_individuales = []


for i in range(len(X_individual)):

    # =======================
    # IMAGEN ORIGINAL

    imagen_original = X_individual[i].astype(np.uint8)
    imagen_original = imagen_original.reshape(28, 28)


    # ======================================
    # CNN

    imagen_cnn = X_individual_cnn[i:i+1]
    inicio = time.perf_counter()

    prob_cnn = modelo_cnn.predict(
        imagen_cnn,
        verbose=0
    )[0][0]

    tiempo_cnn = (time.perf_counter() - inicio)
    pred_cnn = int(prob_cnn >= 0.5)


    # =====================
    # MLP

    imagen_mlp = X_individual_mlp[i:i+1]
    inicio = time.perf_counter()
    pred_mlp = int(modelo_mlp.predict(imagen_mlp)[0])
    tiempo_mlp = (time.perf_counter() - inicio)


    # ================================

    resultados_individuales.append({
        "indice_imagen": i,
        "imagen": imagen_original,
        "etiqueta_real": int(y_individual[i]),

        # CNN
        "prediccion_CNN": pred_cnn,
        "probabilidad_CNN": float(prob_cnn),
        "tiempo_prediccion_CNN_s": tiempo_cnn,

        # MLP
        "prediccion_MLP": pred_mlp,
        "tiempo_prediccion_MLP_s": tiempo_mlp
    })

In [18]:
resultados_individuales_2 = pd.DataFrame(resultados_individuales)

print(
    resultados_individuales_2[
        [
            "indice_imagen",
            "etiqueta_real",
            "prediccion_CNN",
            "probabilidad_CNN",
            "tiempo_prediccion_CNN_s",
            "prediccion_MLP",
            "tiempo_prediccion_MLP_s"
        ]
    ].to_string(index=False)
)

 indice_imagen  etiqueta_real  prediccion_CNN  probabilidad_CNN  tiempo_prediccion_CNN_s  prediccion_MLP  tiempo_prediccion_MLP_s
             0              0               0          0.000057                 0.075277               1                 0.000451
             1              1               1          0.999747                 0.075109               1                 0.000809
             2              0               0          0.000074                 0.101828               1                 0.001045
             3              1               1          0.777666                 0.072212               1                 0.000379
             4              0               0          0.016848                 0.070413               1                 0.000362
             5              0               0          0.000429                 0.068821               1                 0.000362
             6              1               1          0.713448                 0.066247  

## GUARDAR RESULTADOS

In [19]:
resultados_generales_1.to_csv("resultados/resultados_generales_1.csv", index=False)

#==========================
# Guardamos las imagenes individuales

np.savez_compressed(
    "resultados/resultados_individuales_1.npz",

    imagenes= np.stack(resultados_individuales_2["imagen"].values),

    etiquetas_reales= resultados_individuales_2["etiqueta_real"].values,

    predicciones_CNN= resultados_individuales_2["prediccion_CNN"].values,

    probabilidades_CNN= resultados_individuales_2["probabilidad_CNN"].values,

    tiempos_CNN= resultados_individuales_2["tiempo_prediccion_CNN_s"].values,

    predicciones_MLP= resultados_individuales_2["prediccion_MLP"].values,

    tiempos_MLP= resultados_individuales_2["tiempo_prediccion_MLP_s"].values
)


In [22]:
joblib.dump(modelo_cnn, "modelos/test-1_cnn.joblib")
joblib.dump(modelo_mlp, "modelos/test-1_mlp.joblib")

['modelos/test-1_mlp.joblib']